In [ ]:
import socket
import struct
import cv2
import numpy as np
from pynq.overlays.base import BaseOverlay
from pynq.lib.video import *

# --- HDMI Setup ---
base = BaseOverlay("base.bit")
hdmi_out = base.video.hdmi_out
mode = VideoMode(640, 480, 24)
hdmi_out.configure(mode)
hdmi_out.start()
print("[INFO] HDMI output started at 640x480")

In [ ]:
# --- Network Setup ---
HOST = "0.0.0.0"
PORT = 9999

server_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
server_socket.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
server_socket.bind((HOST, PORT))
server_socket.listen(1)
print(f"[INFO] Listening on port {PORT}...")

In [ ]:
# receiver_hdmi.py — run on PYNQ-Z2
conn, addr = server_socket.accept()
print(f"[INFO] Connected by {addr}")

def recv_exact(sock, n):
    buf = b""
    while len(buf) < n:
        chunk = sock.recv(n - len(buf))
        if not chunk:
            return None
        buf += chunk
    return buf

frame_count = 0

try:
    while True:
        header = recv_exact(conn, 5)
        if header is None:
            print("[INFO] Connection closed")
            break

        size, cam_id = struct.unpack(">LB", header)

        data = recv_exact(conn, size)
        if data is None:
            print("[INFO] Connection closed")
            break

        # Decode and resize
        frame = cv2.imdecode(np.frombuffer(data, dtype=np.uint8), cv2.IMREAD_COLOR)
        frame = cv2.resize(frame, (640, 480))

        # BGR → RGB for HDMI
#         frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # Push every frame to HDMI — cam0 and cam1 alternate naturally
        if cam_id == 0:
            frame = cv2.resize(frame, (640, 480))
            
            # Roll the image: take right 60% and put it on the left
            shift = int(640 * 0.4)  # 256 pixels
            frame = np.roll(frame, -shift, axis=1)
            
            frame = np.ascontiguousarray(frame)
            out_frame = hdmi_out.newframe()
            np.copyto(out_frame, frame)
            hdmi_out.writeframe(out_frame)

        frame_count += 1
        if frame_count % 100 == 0:
            print(f"[INFO] Received {frame_count} frames")

except KeyboardInterrupt:
    print("\n[INFO] Stopped")
finally:
    conn.close()
    server_socket.close()
    hdmi_out.stop()
    print(f"[INFO] Total frames received: {frame_count}")